# Tarang v9.5 — PTB-XL PAC Augmentation (Sprint Day 1)

**Objective:** Break the 18-patient MIT-BIH overfitting trap by adding external S-class (PAC/SVPB) data from PTB-XL.

**Honest ceiling:** ~0.35 S F1. The S problem is no longer architectural; it is primarily a data diversity and label noise problem.

---

## What changed vs v9.3 / v9.4

| Lever | v9.3 baseline | v9.4 (Phase A best) | **v9.5** |
|---|---|---|---|
| Window | 130 | 130 (200 also tried, confounded) | **130** |
| Cleaning threshold | 0.95 | 0.95 (best of {0.85, 0.95, 1.05}) | **0.95** |
| N share in SV train | ~0.33 (0.50/0.50 ratio) | same | **0.35** |
| External S data | none | none | **PTB-XL PAC beats** |
| Gate model | v9.3 (frozen) | v9.3 (frozen) | **v9.3 (frozen)** |
| SV model | retrained | retrained per threshold | **retrained with PTB-XL** |

v9.4 Phase B (3-fold CV) confirmed the val/test gap is a **generalization** problem, not a val-size problem. v9.4 Phase C (window 200) was confounded by incomplete SVDB + undertraining, so it is **not** a clean falsification — we keep window 130 here.

## Plan for this notebook (Sprint Day 1)

1. Load v9.3 gate + v9 data checkpoint (MIT-BIH + SVDB beats at 130 samples, 7 RR features)
2. Audit SVDB file count (5-min job — document regardless)
3. Download / locate PTB-XL database
4. Build PTB-XL PAC extraction pipeline:
   - 500 Hz → 250 Hz via `resample_poly` (NOT FFT `resample` — Gibbs artifacts at QRS edges)
   - R-peak detection (`your_pantompkins_detector` placeholder — replace with your actual detector)
   - 130-sample beat windows, rolling-window normalized
   - Prematurity filter at 0.95 (same logic as v9.3 S cleaning, but applied as a KEEP filter — we only want premature beats)
5. **Visual spot-check 50 beats** before trusting bulk extraction (acquisition system differs from MIT-BIH)
6. **Yield checkpoint:** if < 1,000 clean PAC beats, also extract from CPSC2018 before retraining
7. Build augmented SV training set: MIT-BIH train (routed by v9.3 gate) + SVDB S + PTB-XL PAC, with N share = 0.35
8. Train v9.5 SV head (60 epochs, same callbacks as v9.4)
9. Joint threshold sweep on val, eval on MIT-BIH test (untouched)
10. INCART cross-check (optional — runs only if INCART path exists)
11. Save `metrics.json` with the exact schema from the sprint plan

## Critical risks (read before running)

- **Pan-Tompkins transfer:** Your detector was validated on MIT-BIH signals preprocessed through the AD8232/NLMS pipeline. PTB-XL uses a different acquisition system. Threshold parameters may not transfer. The 50-beat spot-check is non-negotiable.
- **Lead mismatch:** PTB-XL is 12-lead; we use lead I only (or I+II if you enable two-channel). MIT-BIH lead II is not lead I — there is morphological divergence.
- **Label noise in PTB-XL:** SVPB/PAC are record-level labels. Not every beat in a SVPB-record is a PAC. The prematurity filter (< 0.95) is what makes the S label trustworthy.


## Setup — paths, constants, load v9.3 gate + v9 data checkpoint

We reuse the v9 data checkpoint (MIT-BIH + SVDB beats, 130 samples, 7 RR features) and the **v9.3 gate model** (frozen — no retraining of the gate this sprint). Only the SV head is retrained.


In [1]:
import os, json, glob, warnings
from collections import Counter
from datetime import datetime

import wfdb, numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns
from scipy.signal import resample_poly
import tensorflow as tf
from tensorflow.keras import regularizers
import sklearn
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import confusion_matrix, f1_score
from sklearn.utils import class_weight

warnings.filterwarnings('ignore')
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    for gpu in gpus: tf.config.experimental.set_memory_growth(gpu, True)

# ── Paths ────────────────────────────────────────────────────────────────────
BASE = 'C:/MMD Public/Hackathons/Team Ocelleon/dataset'
def find_dir(base, *candidates):
    for c in candidates:
        p = os.path.join(base, c)
        if os.path.isdir(p): return p
    for c in candidates:
        for v in [c, c.lower(), c.upper(), c.title()]:
            p = os.path.join(base, v)
            if os.path.isdir(p): return p
    return os.path.join(base, candidates[0])

MITBIH_PATH = find_dir(BASE, 'mit-bih-arrhythmia-database-1.0.0', 'mitbih')
SVDB_PATH   = find_dir(BASE, 'mit-bih-supraventricular-arrhythmia-database-1.0.0', 'svdb')
INCART_PATH = find_dir(BASE, 'incartdb', 'mit-bih-arrhythmia-database-incart', 'st-petersburg-incart-12-lead-arrhythmia-database-1.0.0')
# PTB-XL: point this at the folder that CONTAINS ptbxl_database.csv and records500/ records100/
PTBXL_PATH  = find_dir(BASE, 'ptb-xl-1.0.3', 'ptb-xl', 'ptbxl')

OUTPUTS_V9  = 'outputs_v9'
OUTPUTS_V93 = 'outputs_v93'
OUTPUTS_V95 = 'outputs_v95'
os.makedirs(OUTPUTS_V95, exist_ok=True)

# ── v9.5 hyperparameters (locked from sprint plan) ───────────────────────────
SOURCE_FS, TARGET_FS = 360, 250           # MIT-BIH / SVDB source -> target
PTBXL_FS    = 500                          # PTB-XL high-res lead
INCART_FS   = 257                          # INCART
WINDOW_LEN  = 130                          # sprint plan: 130
WINDOW_PRE  = 65
WINDOW_POST = 65
CLEANING_THRESHOLD = 0.95                  # sprint plan: 0.95 (same as v9.3)
N_SHARE_TARGET     = 0.35                  # sprint plan: 0.35 (changed from v9.4's 0.50/0.50)
SV_SHARE           = 1.0 - N_SHARE_TARGET  # 0.65
SEED = 42
np.random.seed(SEED); tf.random.set_seed(SEED)

# ── AAMI mapping + record splits (identical to v9.3/v9.4) ────────────────────
BEAT_MAP = {
    'N':'N','L':'N','R':'N','e':'N','j':'N',
    'A':'S','a':'S','J':'S','S':'S',
    'V':'V','E':'V','F':'V',
    '/':'Q','f':'Q','Q':'Q',
}
CLASSES_TO_USE = ['N','S','V']
MITBIH_ALL_RECORDS = [
    100,101,102,103,104,105,106,107,108,109,
    111,112,113,114,115,116,117,118,119,121,
    122,123,124,200,201,202,203,205,207,208,
    209,210,212,213,214,215,217,219,220,221,
    222,223,228,230,231,232,233,234
]
MITBIH_TEST_RECORDS = [
    101,106,108,109,112,114,115,116,118,119,
    201,202,203,205,207,208,209,210,217,219,
    221,223,228,231,233,234
]
MITBIH_VAL_RECORDS = [105, 124, 214, 220]
MITBIH_TRAIN_RECORDS = [r for r in MITBIH_ALL_RECORDS
                        if r not in MITBIH_TEST_RECORDS and r not in MITBIH_VAL_RECORDS]
SVDB_RECORDS = [str(i) for i in range(800, 895)]

# ── SVDB audit (sprint plan Day-1 morning) ───────────────────────────────────
svdb_hea_files = sorted(glob.glob(os.path.join(SVDB_PATH, '*.hea')))
svdb_count = len(svdb_hea_files)
print(f"SVDB audit: {svdb_count} .hea files found (expected 78).")
if svdb_count < 78:
    missing = sorted(set(range(800, 895)) - {int(os.path.basename(f).split('.')[0]) for f in svdb_hea_files})
    print(f"  ⚠ Missing records: {missing}")
    print(f"  → Re-download mit-bih-supraventricular-arrhythmia-database-1.0.0 from PhysioNet.")

# ── Load v9 data checkpoint (MIT-BIH + SVDB at 130 samples, 7 RR features) ───
cp = np.load(f'{OUTPUTS_V9}/data_checkpoint.npz', allow_pickle=True)
mb_beats, mb_rr = cp['mb_beats'], cp['mb_rr']
mb_labels, mb_recs = cp['mb_labels'], cp['mb_recs']
sv_beats, sv_rr = cp['sv_beats'], cp['sv_rr']
sv_labels, sv_recs = cp['sv_labels'], cp['sv_recs']

# v9.4 had typos here (`b_beats` instead of `[mb_beats`) — fixed in v9.5
X_all     = np.concatenate([mb_beats, sv_beats], axis=0)
X_rr_all  = np.concatenate([mb_rr,    sv_rr],   axis=0)
y_raw_all = np.concatenate([mb_labels, sv_labels])
recs_all  = np.concatenate([mb_recs,   sv_recs])
le = LabelEncoder(); le.fit(CLASSES_TO_USE)
y_all = le.transform(y_raw_all)
n_idx = int(np.where(le.classes_=='N')[0][0])
s_idx = int(np.where(le.classes_=='S')[0][0])
v_idx = int(np.where(le.classes_=='V')[0][0])

test_recs_set = {f'mitbih_{r}' for r in MITBIH_TEST_RECORDS}
val_recs_set  = {f'mitbih_{r}' for r in MITBIH_VAL_RECORDS}
test_mask  = np.isin(recs_all, list(test_recs_set))
val_mask   = np.isin(recs_all, list(val_recs_set))
train_mask = ~(test_mask | val_mask)

# ── Clean labels at 0.95 (v9.3 rule, identical to v9.4 Phase A baseline) ─────
def clean_labels(prematurity_threshold):
    """Clean S annotations: relabel S beats with prematurity_index >= threshold to N.

    prematurity_index = rr_prev / rr_mean_5  (X_rr_all[:, 6])
    A beat with premat_index >= 0.95 is NOT premature → relabel S→N.
    """
    premat = X_rr_all[:, 6]
    relabel = (y_raw_all == 'S') & (premat >= prematurity_threshold)
    y_raw_clean = y_raw_all.copy()
    y_raw_clean[relabel] = 'N'
    return le.transform(y_raw_clean), y_raw_clean

y_all_clean, y_raw_all_clean = clean_labels(CLEANING_THRESHOLD)
n_test_s_clean  = int(np.sum((y_raw_all_clean == 'S') & test_mask))
n_train_s_clean = int(np.sum((y_raw_all_clean == 'S') & train_mask))
print(f"\nCleaning @ {CLEANING_THRESHOLD}: train S={n_train_s_clean}, test S={n_test_s_clean}")

# ── RR normalization (stats from MIT-BIH+SVDB train only — PTB-XL normalized with same stats) ─
RR_MEAN = X_rr_all[train_mask].mean(axis=0)
RR_STD  = X_rr_all[train_mask].std(axis=0); RR_STD[RR_STD < 1e-8] = 1e-8
X_rr_all_norm = (X_rr_all - RR_MEAN) / RR_STD

# ── Load v9.3 gate (frozen for this sprint) ──────────────────────────────────
gate_v93 = tf.keras.models.load_model(f'{OUTPUTS_V93}/gate_v93.keras', compile=False)
sv_v93   = tf.keras.models.load_model(f'{OUTPUTS_V93}/sv_v93.keras', compile=False)
print(f"Loaded v9.3 gate ({gate_v93.count_params():,} params)")
print(f"Loaded v9.3 SV   ({sv_v93.count_params():,} params) — used as baseline for comparison")

# v9.3 thresholds (used for v9.3 baseline eval; v9.5 will get its own sweep)
V93_GATE_THR, V93_V_THR, V93_S_THR = 0.100, 0.20, 0.50

# ── Helpers (reused from v9.4, unchanged) ────────────────────────────────────
def augment_batch(X_c, X_rr_c, class_name, n_copies, rng=None):
    if rng is None: rng = np.random.default_rng(SEED)
    n = len(X_c)
    if n == 0 or n_copies == 0:
        return (np.empty((0,)+X_c.shape[1:], dtype=np.float32),
                np.empty((0,)+X_rr_c.shape[1:], dtype=np.float32))
    out_X = np.repeat(X_c, n_copies, axis=0)
    out_rr = np.repeat(X_rr_c, n_copies, axis=0)
    shift = rng.integers(-3, 4, size=len(out_X))
    out_X_aug = np.empty_like(out_X)
    for i, s in enumerate(shift):
        if s > 0: out_X_aug[i, :-s] = out_X[i, s:]; out_X_aug[i, -s:] = out_X[i, -1:]
        elif s < 0: out_X_aug[i, -s:] = out_X[i, :s]; out_X_aug[i, :-s] = out_X[i, :1]
        else: out_X_aug[i] = out_X[i]
    amp = rng.uniform(0.85, 1.15, size=(len(out_X), 1, 1)).astype(np.float32)
    out_X_aug *= amp
    out_X_aug += rng.normal(0, 0.02, size=out_X_aug.shape).astype(np.float32)
    if class_name == 'S':
        out_rr[:, 0] *= rng.uniform(0.55, 0.85, size=len(out_rr))
        out_rr[:, 1] *= rng.uniform(1.10, 1.40, size=len(out_rr))
        out_rr[:, 4] *= rng.uniform(1.20, 2.00, size=len(out_rr))
        out_rr[:, 6] = out_rr[:, 0] / np.maximum(out_rr[:, 3], 1e-4)
        out_rr[:, 5] = out_rr[:, 1] / np.maximum(out_rr[:, 3], 1e-4)
    elif class_name == 'V':
        out_rr[:, 1] *= rng.uniform(1.20, 1.60, size=len(out_rr))
        out_rr[:, 4] *= rng.uniform(1.10, 1.80, size=len(out_rr))
        out_rr[:, 6] = out_rr[:, 0] / np.maximum(out_rr[:, 3], 1e-4)
        out_rr[:, 5] = out_rr[:, 1] / np.maximum(out_rr[:, 3], 1e-4)
    return out_X_aug, out_rr

def build_sv_model(ecg_shape=(WINDOW_LEN,2), rr_shape=(7,)):
    ecg_in = tf.keras.Input(shape=ecg_shape, name='ecg_input')
    x = tf.keras.layers.Reshape((WINDOW_LEN,2,1))(ecg_in)
    for f,k,d in [(16,7,0.1),(32,5,0.1),(48,5,0.15),(48,3,0.0)]:
        x = tf.keras.layers.Conv2D(f,(k,1),padding='same',use_bias=False,
                kernel_regularizer=regularizers.l2(1e-4))(x)
        x = tf.keras.layers.BatchNormalization()(x)
        x = tf.keras.layers.Activation('relu')(x)
        if k >= 5: x = tf.keras.layers.MaxPooling2D((2,1))(x); x = tf.keras.layers.SpatialDropout2D(d)(x)
    x = tf.keras.layers.GlobalAveragePooling2D()(x)
    rr_in = tf.keras.Input(shape=rr_shape, name='rr_input')
    r = tf.keras.layers.Dense(16, activation='relu', kernel_regularizer=regularizers.l2(1e-4))(rr_in)
    r = tf.keras.layers.Dropout(0.2)(r)
    r = tf.keras.layers.Dense(8, activation='relu', kernel_regularizer=regularizers.l2(1e-4))(r)
    m = tf.keras.layers.Concatenate()([x, r])
    m = tf.keras.layers.Dense(32, use_bias=False)(m)
    m = tf.keras.layers.BatchNormalization()(m)
    m = tf.keras.layers.Activation('relu')(m)
    m = tf.keras.layers.Dropout(0.35)(m)
    v = tf.keras.layers.Dense(1, activation='sigmoid', name='v_head')(m)
    s = tf.keras.layers.Dense(1, activation='sigmoid', name='s_head')(m)
    return tf.keras.Model(inputs=[ecg_in,rr_in], outputs=[v,s])

def joint_threshold_sweep(gate_probs, v_probs, s_probs, y_true):
    GATE_THRS = np.arange(0.10, 0.71, 0.025)
    V_THRS    = np.arange(0.10, 0.95, 0.05)
    S_THRS    = np.arange(0.10, 0.95, 0.05)
    rows = []
    y_t = y_true.astype(np.int8)
    for g_thr in GATE_THRS:
        routed = gate_probs > g_thr
        if routed.sum() == 0: continue
        for v_thr in V_THRS:
            v_claim = routed & (v_probs > v_thr)
            for s_thr in S_THRS:
                s_claim = routed & (~v_claim) & (s_probs > s_thr)
                y_p = np.full_like(y_t, n_idx)
                y_p[v_claim] = v_idx; y_p[s_claim] = s_idx
                flat = (y_t * 3 + y_p).astype(np.int32)
                cm = np.bincount(flat, minlength=9).reshape(3, 3)
                f1s, recalls, precisions = [], [], []
                for i in range(3):
                    tp = int(cm[i,i]); fn = int(cm[i,:].sum()-tp); fp = int(cm[:,i].sum()-tp)
                    rec = tp/max(tp+fn,1); prec = tp/max(tp+fp,1)
                    f1s.append(2*prec*rec/max(prec+rec,1e-7))
                    recalls.append(rec); precisions.append(prec)
                rows.append({'gate_thr':float(g_thr),'v_thr':float(v_thr),'s_thr':float(s_thr),
                              'macro_f1':float(np.mean(f1s)),
                              's_recall':recalls[s_idx],'s_precision':precisions[s_idx],
                              'v_recall':recalls[v_idx],'n_precision':precisions[n_idx]})
    return pd.DataFrame(rows)

def eval_cascade(gate_model, sv_model, X, X_rr_norm, y_true, gate_thr, v_thr, s_thr, le, name=""):
    gate_probs = gate_model.predict([X, X_rr_norm], batch_size=256, verbose=0).flatten()
    gate_pass = gate_probs > gate_thr
    y_pred = np.full(len(y_true), n_idx, dtype=int)
    if gate_pass.any():
        v_p, s_p = sv_model.predict([X[gate_pass], X_rr_norm[gate_pass]], batch_size=256, verbose=0)
        v_p = v_p.flatten(); s_p = s_p.flatten()
        routed = np.full(len(v_p), n_idx, dtype=int)
        v_fire = v_p > v_thr; routed[v_fire] = v_idx
        s_fire = (~v_fire) & (s_p > s_thr); routed[s_fire] = s_idx
        y_pred[gate_pass] = routed
    cm = confusion_matrix(y_true, y_pred, labels=[n_idx, s_idx, v_idx])
    m = {'set': name, 'macro_f1': float(f1_score(y_true, y_pred, average='macro', zero_division=0))}
    for i, cls in enumerate(le.classes_):
        tp = int(cm[i,i]); fn = int(cm[i,:].sum()-tp); fp = int(cm[:,i].sum()-tp)
        total = int(cm[i,:].sum())
        rec = tp/max(total,1); prec = tp/max(tp+fp,1)
        m[f'{cls}_f1'] = 2*prec*rec/max(prec+rec,1e-7)
        m[f'{cls}_recall'] = rec; m[f'{cls}_precision'] = prec; m[f'{cls}_total'] = total
    return m

# v9.3 baseline metrics on cleaned test (for comparison table)
v93_test_metrics = eval_cascade(gate_v93, sv_v93,
    X_all[test_mask], X_rr_all_norm[test_mask], y_all_clean[test_mask],
    V93_GATE_THR, V93_V_THR, V93_S_THR, le, "v9.3 baseline (0.95 cleaning)")
print(f"\nv9.3 baseline on cleaned test: macro_f1={v93_test_metrics['macro_f1']:.4f}, "
      f"S F1={v93_test_metrics['S_f1']:.3f}, S recall={v93_test_metrics['S_recall']:.3f}, "
      f"S prec={v93_test_metrics['S_precision']:.3f}, V F1={v93_test_metrics['V_f1']:.3f}")

ALL_RESULTS = [{'experiment': 'v9.3_baseline', **v93_test_metrics}]
print("\nSetup complete. Ready for PTB-XL extraction.")


SVDB audit: 78 .hea files found (expected 78).

Cleaning @ 0.95: train S=10604, test S=566
Loaded v9.3 gate (28,633 params)
Loaded v9.3 SV   (20,090 params) — used as baseline for comparison

v9.3 baseline on cleaned test: macro_f1=0.5592, S F1=0.199, S recall=0.157, S prec=0.271, V F1=0.567

Setup complete. Ready for PTB-XL extraction.


## PTB-XL PAC extraction pipeline

PTB-XL is **record-level** annotated (no beat-level R-peak annotations). We must:
1. Load the 500 Hz signal (lead I)
2. Resample 500 → 250 Hz with `resample_poly` (polyphase — no Gibbs artifacts at QRS edges, unlike FFT-based `resample`)
3. Run R-peak detection
4. Extract 130-sample windows (65 pre / 65 post), rolling-window normalized
5. Compute the same 7 RR features used by MIT-BIH/SVDB
6. **Keep** only beats where `prematurity_index = rr_prev / rr_mean_5 < 0.95` (this is the inverse of the v9.3 S-cleaning rule — we want premature beats, which are the real PACs)
7. Tag all surviving beats as 'S' (these come from SVPB/PAC records, and the prematurity filter confirms they are PACs)

**Critical risk — replace `your_pantompkins_detector` with your real detector.** The fallback `wfdb.processing.xqrs_detect` is signal-agnostic and works across acquisition systems, but it is NOT your validated detector. The sprint plan explicitly warns that your threshold parameters may not transfer from MIT-BIH/AD8232 to PTB-XL.


In [2]:
# ── Signal preprocessing helpers (reused from v9.4 Phase C, kept identical) ───
def rolling_window_normalize(signal, fs, window_seconds=30):
    """30-second rolling z-score normalization (matches v9 extraction)."""
    ws = int(window_seconds * fs)
    s = pd.Series(signal.astype(np.float64))
    roll = s.rolling(window=ws, min_periods=1)
    mean = roll.mean(); std = roll.std(ddof=0).fillna(0).clip(lower=1e-8)
    return ((s - mean) / std).values.astype(np.float32)

def compute_rr_features(peaks_sec, beat_idx):
    """7 RR features — IDENTICAL to v9.4 cell-8 compute_rr_features so PTB-XL beats
    live in the same feature space as MIT-BIH/SVDB beats.

    Index 6 (rr_prev / rr_mean_5) is the prematurity_index used by clean_labels().
    """
    n = len(peaks_sec); i = beat_idx
    prev_idx = max(0, i - 1); next_idx = min(n - 1, i + 1)
    rr_prev = peaks_sec[i] - peaks_sec[prev_idx]
    rr_next = peaks_sec[next_idx] - peaks_sec[i]
    rr_ratio = rr_prev / max(rr_next, 1e-4)
    lo, hi = max(0, i - 2), min(n - 1, i + 2)
    local_rrs = np.diff(peaks_sec[lo:hi+1]).astype(np.float32)
    if len(local_rrs) == 0: rr_mean_5, rr_std_5 = rr_prev, 0.0
    else: rr_mean_5 = float(np.mean(local_rrs)); rr_std_5 = float(np.std(local_rrs))
    return np.array([rr_prev, rr_next, rr_ratio, rr_mean_5, rr_std_5,
                      rr_next/max(rr_mean_5,1e-4), rr_prev/max(rr_mean_5,1e-4)], dtype=np.float32)

# ── R-peak detector ──────────────────────────────────────────────────────────
# ⚠ REPLACE THIS with your real Pan-Tompkins implementation.
# The xqrs_detect fallback is signal-agnostic but NOT your validated detector.
# Your detector was tuned on MIT-BIH signals preprocessed through AD8232/NLMS;
# PTB-XL uses a different acquisition system and your thresholds may not transfer.
def your_pantompkins_detector(signal_250, fs=250):
    """Drop-in placeholder. Returns R-peak sample indices in `signal_250`.

    Replace the body below with your real Pan-Tompkins implementation. The
    interface contract is: 1-D numpy array in, 1-D numpy array of int peak
    indices out (same units as wfdb.rdann().sample).
    """
    try:
        # Fallback: WFDB's XQRS — robust across acquisition systems but ~5× slower
        # than a hand-tuned Pan-Tompkins.
        peaks = wfdb.processing.xqrs_detect(sig=signal_250.astype(np.float64), fs=fs)
        return np.asarray(peaks, dtype=np.int64)
    except Exception as e:
        print(f"  [xqrs failed] {e}")
        return np.array([], dtype=np.int64)

# ── PTB-XL PAC extraction ────────────────────────────────────────────────────
def extract_ptbxl_pac_beats(record_id, window=WINDOW_LEN, fs_target=TARGET_FS,
                            fs_source=PTBXL_FS, prematurity_threshold=CLEANING_THRESHOLD,
                            two_channel=False, verbose=False):
    """Extract PAC beats from a single PTB-XL record.

    Returns (beats [N, window, 2], rr [N, 7]) — beats is 2-channel even if
    two_channel=False (we duplicate lead I to channel 2 to match the SV model input).
    """
    # PTB-XL directory layout: records500/{ecg_id//1000:05d}/{ecg_id:05d}_hr
    sub = f"{record_id // 1000 * 1000:05d}"
    path = f'{PTBXL_PATH}/records500/{sub}/{record_id:05d}_hr'
    try:
        record = wfdb.rdrecord(path)
    except Exception as e:
        if verbose: print(f"  [skip] ptbxl_{record_id}: {e}")
        return np.empty((0, window, 2), dtype=np.float32), np.empty((0, 7), dtype=np.float32)

    # Lead I (and II if two_channel). PTB-XL stores lead I at column 0, II at column 1.
    p_signal = np.asarray(record.p_signal, dtype=np.float64)
    sig0 = p_signal[:, 0]
    sig1 = p_signal[:, 1] if (two_channel and p_signal.shape[1] > 1) else sig0

    # 500 -> 250 Hz polyphase (NOT FFT resample — sprint plan rule)
    sig0_250 = resample_poly(sig0, up=1, down=2).astype(np.float32)
    sig1_250 = resample_poly(sig1, up=1, down=2).astype(np.float32)

    # Rolling-window normalize (matches v9 extraction)
    ecg_ch0 = rolling_window_normalize(sig0_250, fs_target)
    ecg_ch1 = rolling_window_normalize(sig1_250, fs_target)
    if not two_channel:
        ecg_ch1 = ecg_ch0  # duplicate to keep tensor shape (WINDOW_LEN, 2)

    # R-peak detection on lead I
    r_peaks = your_pantompkins_detector(ecg_ch0, fs=fs_target)
    if len(r_peaks) < 5:
        return np.empty((0, window, 2), dtype=np.float32), np.empty((0, 7), dtype=np.float32)

    peaks_sec = r_peaks / fs_target
    half = window // 2
    beats, rrs = [], []
    for i, peak in enumerate(r_peaks):
        if peak - half < 0 or peak + half >= len(ecg_ch0):
            continue
        rr_feat = compute_rr_features(peaks_sec, i)
        prematurity_index = float(rr_feat[6])  # rr_prev / rr_mean_5
        # KEEP only premature beats (the inverse of the v9.3 S-cleaning rule).
        # This is what makes the 'S' label trustworthy on PTB-XL.
        if prematurity_index >= prematurity_threshold:
            continue
        beat = np.stack([ecg_ch0[peak - half: peak + half],
                         ecg_ch1[peak - half: peak + half]], axis=-1).astype(np.float32)
        beats.append(beat); rrs.append(rr_feat)
    if not beats:
        return np.empty((0, window, 2), dtype=np.float32), np.empty((0, 7), dtype=np.float32)
    return np.stack(beats), np.stack(rrs)

print("PTB-XL extraction functions defined.")
print(f"  Window: {WINDOW_LEN} samples ({WINDOW_PRE} pre / {WINDOW_POST} post @ {TARGET_FS} Hz)")
print(f"  Cleaning / prematurity threshold: {CLEANING_THRESHOLD}")
print(f"  ⚠ Replace your_pantompkins_detector() with your real implementation before bulk extraction.")


PTB-XL extraction functions defined.
  Window: 130 samples (65 pre / 65 post @ 250 Hz)
  Cleaning / prematurity threshold: 0.95
  ⚠ Replace your_pantompkins_detector() with your real implementation before bulk extraction.


In [3]:
# ── Locate PTB-XL metadata ────────────────────────────────────────────────────
ptbxl_csv = os.path.join(PTBXL_PATH, 'ptbxl_database.csv')
if not os.path.isfile(ptbxl_csv):
    raise FileNotFoundError(
        f"PTB-XL database CSV not found at {ptbxl_csv}.\n"
        f"Download from https://physionet.org/files/ptb-xl/1.0.3/ "
        f"(~2.3 GB) and unzip into {PTBXL_PATH}."
    )

ptbxl_db = pd.read_csv(ptbxl_csv, index_col='ecg_id')

# Filter for SVPB / PAC records (PTB-XL uses SCP codes; SVPB covers PAC)
pac_mask = ptbxl_db['scp_codes'].str.contains('SVPB', na=False) | \
           ptbxl_db['scp_codes'].str.contains('PAC',  na=False)
pac_records = ptbxl_db[pac_mask]
print(f"PTB-XL PAC/SVPB records available: {len(pac_records)}")
print(f"PTB-XL total records in CSV: {len(ptbxl_db)}")

# ── Bulk extraction ──────────────────────────────────────────────────────────
# Cap at 1000 records (sprint plan). Adjust up if yield < 1,000 beats.
MAX_RECORDS = 1000
record_ids = list(pac_records.index[:MAX_RECORDS])
print(f"\nExtracting from up to {len(record_ids)} records (cap={MAX_RECORDS})...")

all_pac_beats, all_pac_rr = [], []
per_record_yield = []
t0 = pd.Timestamp.now()
for j, rid in enumerate(record_ids):
    beats, rrs = extract_ptbxl_pac_beats(int(rid))
    if len(beats) > 0:
        all_pac_beats.append(beats); all_pac_rr.append(rrs)
        per_record_yield.append((rid, len(beats)))
    if (j + 1) % 50 == 0:
        n_so_far = sum(len(b) for b in all_pac_beats)
        elapsed = (pd.Timestamp.now() - t0).total_seconds()
        rate = (j + 1) / max(elapsed, 1e-3)
        eta = (len(record_ids) - j - 1) / max(rate, 1e-3)
        print(f"  [{j+1:>4}/{len(record_ids)}] beats={n_so_far:>5}  "
              f"rate={rate:.1f} rec/s  ETA={eta/60:.1f} min")

if all_pac_beats:
    ptbxl_beats = np.concatenate(all_pac_beats, axis=0)
    ptbxl_rr    = np.concatenate(all_pac_rr,    axis=0)
else:
    ptbxl_beats = np.empty((0, WINDOW_LEN, 2), dtype=np.float32)
    ptbxl_rr    = np.empty((0, 7), dtype=np.float32)
ptbxl_labels = np.array(['S'] * len(ptbxl_beats), dtype=object)
ptbxl_recs   = np.array([f'ptbxl_{rid}' for rid, _ in per_record_yield],
                         dtype=object).repeat([n for _, n in per_record_yield])

print(f"\n{'='*80}")
print(f"PTB-XL EXTRACTION COMPLETE")
print(f"{'='*80}")
print(f"  Records processed: {len(record_ids)}")
print(f"  Records yielded beats: {len(per_record_yield)}")
print(f"  Total clean PAC beats: {len(ptbxl_beats)}")
print(f"  Per-record mean yield: {len(ptbxl_beats) / max(len(per_record_yield), 1):.1f} beats/rec")

# ── Yield checkpoint (sprint plan rule) ──────────────────────────────────────
YIELD_FLOOR = 1000
if len(ptbxl_beats) < YIELD_FLOOR:
    print(f"\n  ⚠ YIELD CHECKPOINT FAILED: {len(ptbxl_beats)} < {YIELD_FLOOR} clean PAC beats.")
    print(f"    Per sprint plan: also extract from CPSC2018 before retraining.")
    print(f"    Do NOT train v9.5 on fewer than {YIELD_FLOOR} external S beats — signal will be too weak.")
    print(f"    Diagnostics:")
    print(f"      - Visually inspect 50 beats (next cell) — is QRS morphology preserved?")
    print(f"      - Check xqrs_detect output on 3-5 records — are R-peak locations sane?")
    print(f"      - If xqrs is failing on PTB-XL, try records100/ (100 Hz) instead of records500/.")
    print(f"      - If still failing, the S morphology may be unlearnable on single-lead.")
else:
    print(f"\n  ✓ Yield checkpoint passed: {len(ptbxl_beats)} ≥ {YIELD_FLOOR} clean PAC beats.")
    print(f"    Proceeding to visual spot-check (next cell).")

# Save extracted PTB-XL beats for reuse / debugging
np.savez_compressed(f'{OUTPUTS_V95}/ptbxl_pac_beats.npz',
                    beats=ptbxl_beats, rr=ptbxl_rr,
                    labels=ptbxl_labels, recs=ptbxl_recs)
print(f"  Saved: {OUTPUTS_V95}/ptbxl_pac_beats.npz")


FileNotFoundError: PTB-XL database CSV not found at C:/MMD Public/Hackathons/Team Ocelleon/dataset\ptb-xl\ptbxl_database.csv.
Download from https://physionet.org/files/ptb-xl/1.0.3/ (~2.3 GB) and unzip into C:/MMD Public/Hackathons/Team Ocelleon/dataset\ptb-xl.

## Visual spot-check — 50 PTB-XL PAC beats

**Do not skip this cell.** The sprint plan explicitly requires this:

> Your Pan-Tompkins was validated on MIT-BIH signals preprocessed through your AD8232/NLMS pipeline. PTB-XL signals come from a different acquisition system. Your threshold parameters may not transfer. Visually spot-check 50 extracted beats before running bulk extraction.

Look for:
- **QRS alignment**: R-peak should be at sample 65 (center of the 130-sample window). If QRS is shifted left/right, R-peak detection is off.
- **P-wave visibility**: PACs should have a discernible P-wave before the QRS (different morphology from sinus P-wave is the whole point).
- **Compensatory pause**: Post-QRS RR should be longer than pre-QRS RR.
- **Baseline wander**: If the rolling-window normalize is working, baseline should be flat near zero.
- **Gibbs artifacts**: Sharp ringing at QRS onset/offset means `resample` was used instead of `resample_poly`. We use `resample_poly` — should be clean.

If > 10% of beats look broken, fix the detector or resampler BEFORE training.


In [ ]:
N_PLOT = 50
n_avail = len(ptbxl_beats)
if n_avail == 0:
    print("No PTB-XL beats to plot. Fix extraction before continuing.")
else:
    idx = np.random.default_rng(SEED).choice(n_avail, size=min(N_PLOT, n_avail), replace=False)
    n_cols = 10
    n_rows = int(np.ceil(len(idx) / n_cols))
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(n_cols * 2.2, n_rows * 1.8),
                              constrained_layout=True)
    axes = np.atleast_2d(axes)
    t = np.arange(WINDOW_LEN) / TARGET_FS * 1000  # ms
    for k, ax in enumerate(axes.flat):
        if k >= len(idx): ax.axis('off'); continue
        b = ptbxl_beats[idx[k]]
        rr = ptbxl_rr[idx[k]]
        ax.plot(t, b[:, 0], color='#1f77b4', lw=1.0)
        ax.plot(t, b[:, 1], color='#d62728', lw=0.6, alpha=0.5)
        ax.axvline(WINDOW_PRE / TARGET_FS * 1000, color='k', lw=0.5, ls='--', alpha=0.4)
        ax.set_xticks([]); ax.set_yticks([])
        ax.set_title(f"prem={rr[6]:.2f}", fontsize=7)
    fig.suptitle(f"PTB-XL PAC spot-check — {len(idx)} beats (lead I blue, lead II red if 2-ch)\n"
                  f"R-peak should be at center dashed line (sample 65 / 260 ms)",
                  fontsize=10)
    plt.savefig(f'{OUTPUTS_V95}/ptbxl_spotcheck.png', dpi=120, bbox_inches='tight')
    plt.show()
    print(f"Saved: {OUTPUTS_V95}/ptbxl_spotcheck.png")
    print(f"\nInspect the plot. If QRS morphology looks broken, fix the detector / resampler before continuing.")


## Build augmented SV training set

Pipeline:
1. Route MIT-BIH+SVDB train beats through the **frozen v9.3 gate** (threshold 0.10) — same as v9.4 Phase A
2. Add **all** PTB-XL PAC beats to the routed set (they are pre-filtered to premature-only, all labeled S — no gate routing needed since they are guaranteed non-N)
3. Apply v9.3 cleaning (already done at setup — `y_all_clean`)
4. Compute `target_sv = max(n_s, n_v)` after PTB-XL augmentation
5. Compute `target_n` from `N_SHARE_TARGET = 0.35`:
   - If total = T, then N = 0.35·T, S+V = 0.65·T, balanced S=V → `target_sv = 0.325·T`
   - Solving: `target_n = (0.35 / 0.65) · 2 · target_sv ≈ 1.077 · target_sv`
6. Augment with `augment_batch` (same shifts / amplitude / noise as v9.4)
7. Train-val split: val is the v9.3 4-patient val set, gate-routed (NO PTB-XL in val — PTB-XL is train-only)

**Key constraint from sprint plan:** PTB-XL beats go ONLY into SV training. MIT-BIH test stays untouched. INCART is fully held-out.


In [ ]:
# ── Splits (use y_all_clean — cleaning already applied) ──────────────────────
y_train = y_all_clean[train_mask]; y_train_raw = y_raw_all_clean[train_mask]
y_val   = y_all_clean[val_mask]
y_test  = y_all_clean[test_mask]

X_train = X_all[train_mask]; X_rr_train = X_rr_all_norm[train_mask]
X_val   = X_all[val_mask];   X_rr_val   = X_rr_all_norm[val_mask]
X_test  = X_all[test_mask];  X_rr_test  = X_rr_all_norm[test_mask]

# ── Route MIT-BIH+SVDB train through v9.3 gate ───────────────────────────────
gate_probs_train = gate_v93.predict([X_train, X_rr_train], batch_size=256, verbose=0).flatten()
routed_mask = gate_probs_train > V93_GATE_THR
sv_X = X_train[routed_mask]; sv_rr = X_rr_train[routed_mask]
sv_y = y_train[routed_mask]; sv_y_raw = y_train_raw[routed_mask]

print(f"v9.3 gate routed {routed_mask.sum()} / {len(X_train)} train beats "
      f"(thr={V93_GATE_THR:.3f})")

# ── Append PTB-XL PAC beats (all S, normalize RR with MIT-BIH train stats) ───
if len(ptbxl_beats) > 0:
    ptbxl_rr_norm = (ptbxl_rr - RR_MEAN) / RR_STD
    ptbxl_y = np.full(len(ptbxl_beats), s_idx, dtype=sv_y.dtype)
    sv_X   = np.concatenate([sv_X,   ptbxl_beats.astype(np.float32)], axis=0)
    sv_rr  = np.concatenate([sv_rr,  ptbxl_rr_norm.astype(np.float32)], axis=0)
    sv_y   = np.concatenate([sv_y,   ptbxl_y], axis=0)
    sv_y_raw = np.concatenate([sv_y_raw, np.array(['S']*len(ptbxl_beats), dtype=object)])
    print(f"Added {len(ptbxl_beats)} PTB-XL PAC beats (all S) to SV training set.")
else:
    print("⚠ No PTB-XL beats available — training on MIT-BIH+SVDB only (degenerates to v9.3).")

# ── Per-class counts AFTER PTB-XL augmentation ───────────────────────────────
n_per = Counter(sv_y)
n_n = n_per.get(n_idx, 0); n_s = n_per.get(s_idx, 0); n_v = n_per.get(v_idx, 0)
target_sv = max(n_s, n_v)
# N share = 0.35, SV share = 0.65, balanced S=V=target_sv
# → target_n / (target_n + 2*target_sv) = 0.35
# → target_n = (0.35/0.65) * 2 * target_sv
target_n = int(round((N_SHARE_TARGET / SV_SHARE) * 2 * target_sv))
print(f"\nAfter augmentation: N={n_n}, S={n_s}, V={n_v}")
print(f"  target_sv (S=V balanced) = {target_sv}")
print(f"  target_n  (N_SHARE={N_SHARE_TARGET:.2f}) = {target_n}")
print(f"  Expected N share post-aug = "
      f"{target_n / (target_n + 2*target_sv):.3f} (target {N_SHARE_TARGET:.3f})")

# ── Augment to targets ───────────────────────────────────────────────────────
sv_X_list = [sv_X]; sv_rr_list = [sv_rr]; sv_y_list = [sv_y]
for class_idx, class_name in enumerate(le.classes_):
    n_have = n_per.get(class_idx, 0)
    n_need_target = target_n if class_idx == n_idx else target_sv
    n_need = max(0, n_need_target - n_have)
    if n_need == 0 or n_have == 0:
        print(f"  {class_name}: have={n_have}, need={n_need_target} (skip — already at/above target or no seed beats)")
        continue
    mask = sv_y == class_idx
    X_c, rr_c = sv_X[mask], sv_rr[mask]
    n_copies = max(1, n_need // n_have + (1 if n_need % n_have else 0))
    n_copies = min(n_copies, 10)
    X_aug, rr_aug = augment_batch(X_c, rr_c, class_name, n_copies)
    y_aug = np.full(len(X_aug), class_idx, dtype=sv_y.dtype)
    sv_X_list.append(X_aug); sv_rr_list.append(rr_aug); sv_y_list.append(y_aug)
    print(f"  {class_name}: have={n_have}, need={n_need_target}, copies={n_copies}, "
          f"augmented={len(X_aug)} (total now {n_have + len(X_aug)})")

sv_X_aug = np.concatenate(sv_X_list)
sv_rr_aug = np.concatenate(sv_rr_list)
sv_y_aug = np.concatenate(sv_y_list)
perm = np.random.permutation(len(sv_X_aug))
sv_X_aug = sv_X_aug[perm]; sv_rr_aug = sv_rr_aug[perm]; sv_y_aug = sv_y_aug[perm]

# Final composition audit
final_counts = Counter(sv_y_aug)
print(f"\nFinal SV training set composition:")
for ci, cn in enumerate(le.classes_):
    share = final_counts.get(ci, 0) / len(sv_y_aug)
    print(f"  {cn}: {final_counts.get(ci, 0):>6} ({share:.3f})")
print(f"  total: {len(sv_y_aug)}")

# ── Val (gate-routed, NO PTB-XL) ─────────────────────────────────────────────
gate_probs_val = gate_v93.predict([X_val, X_rr_val], batch_size=256, verbose=0).flatten()
routed_val = gate_probs_val > V93_GATE_THR
sv_X_val = X_val[routed_val]; sv_rr_val = X_rr_val[routed_val]; sv_y_val = y_val[routed_val]
print(f"\nVal (gate-routed): {len(sv_X_val)} beats, S={int(np.sum(sv_y_val==s_idx))}, V={int(np.sum(sv_y_val==v_idx))}")

# Persist the augmented training set for reproducibility
np.savez_compressed(f'{OUTPUTS_V95}/sv_train_augmented.npz',
                    X=sv_X_aug, rr=sv_rr_aug, y=sv_y_aug,
                    X_val=sv_X_val, rr_val=sv_rr_val, y_val=sv_y_val)
print(f"Saved: {OUTPUTS_V95}/sv_train_augmented.npz")


## Train v9.5 SV head

Same architecture as v9.3 / v9.4 (`build_sv_model`). Same training recipe (60 epochs, Adam 1e-3, class-weighted binary crossentropy on V and S heads, combined-score callback, early stopping, ReduceLROnPlateau). Only difference: the training set now includes PTB-XL PAC beats and uses N_SHARE = 0.35.

The v9.3 gate is **frozen** — we do not retrain it this sprint.


In [ ]:
# ── Build v9.5 SV (identical architecture to v9.3) ───────────────────────────
sv_v95 = build_sv_model()

y_v_aug = (sv_y_aug == v_idx).astype(np.float32)
y_s_aug = (sv_y_aug == s_idx).astype(np.float32)
y_v_val = (sv_y_val == v_idx).astype(np.float32)
y_s_val = (sv_y_val == s_idx).astype(np.float32)
cw_v = class_weight.compute_class_weight('balanced', classes=np.array([0,1]), y=y_v_aug.astype(int))
cw_s = class_weight.compute_class_weight('balanced', classes=np.array([0,1]), y=y_s_aug.astype(int))
sw_v = np.where(y_v_aug==1, cw_v[1], cw_v[0]).astype(np.float32)
sw_s = np.where(y_s_aug==1, cw_s[1], cw_s[0]).astype(np.float32)

class CB(tf.keras.callbacks.Callback):
    def __init__(self, vd, yv, ys): self.vd=vd; self.yv=yv; self.ys=ys
    def on_epoch_end(self, e, logs=None):
        logs = logs or {}
        v,s = self.model.predict(self.vd, verbose=0)
        v=v.flatten(); s=s.flatten()
        vp=(v>0.5).astype(int); sp=(s>0.5).astype(int)
        def pr(yt,yp):
            tp=np.sum((yt==1)&(yp==1));fp=np.sum((yt==0)&(yp==1));fn=np.sum((yt==1)&(yp==0))
            return tp/max(tp+fp,1),tp/max(tp+fn,1)
        vpr,vr=pr(self.yv,vp); spr,sr=pr(self.ys,sp)
        logs['val_combined_score']=float(0.5*(vr+sr))

sv_v95.compile(optimizer=tf.keras.optimizers.Adam(1e-3),
    loss={'v_head':'binary_crossentropy','s_head':'binary_crossentropy'},
    metrics={'v_head':[tf.keras.metrics.AUC(name='auc')],'s_head':[tf.keras.metrics.AUC(name='auc')]})

print(f"Training v9.5 SV head — {len(sv_X_aug)} samples, "
      f"N={int(np.sum(sv_y_aug==n_idx))} S={int(np.sum(sv_y_aug==s_idx))} V={int(np.sum(sv_y_aug==v_idx))}")
print(f"  Architecture params: {sv_v95.count_params():,}")
print(f"  Epochs=60, batch=256, early-stop patience=12")

history = sv_v95.fit([sv_X_aug, sv_rr_aug], {'v_head':y_v_aug, 's_head':y_s_aug},
    sample_weight={'v_head':sw_v, 's_head':sw_s},
    validation_data=([sv_X_val, sv_rr_val], {'v_head':y_v_val, 's_head':y_s_val}),
    epochs=60, batch_size=256,
    callbacks=[CB([sv_X_val, sv_rr_val], y_v_val, y_s_val),
        tf.keras.callbacks.ModelCheckpoint(f'{OUTPUTS_V95}/sv_v95.keras',
            monitor='val_combined_score', mode='max', save_best_only=True, verbose=1),
        tf.keras.callbacks.EarlyStopping(monitor='val_combined_score', mode='max',
            patience=12, restore_best_weights=True, verbose=1),
        tf.keras.callbacks.ReduceLROnPlateau(monitor='val_combined_score', mode='max',
            factor=0.5, patience=5, min_lr=1e-6, verbose=1)],
    verbose=2)

# Reload the saved best model so the artifact on disk matches what we evaluate
sv_v95 = tf.keras.models.load_model(f'{OUTPUTS_V95}/sv_v95.keras', compile=False)
print(f"\n✓ Saved: {OUTPUTS_V95}/sv_v95.keras")
print(f"✓ Loaded best weights (val_combined_score={max(history.history.get('val_combined_score', [0])):.4f})")


## Joint threshold sweep on val + eval on MIT-BIH test

Sweep gate/V/S thresholds on the v9.3 4-patient val set, pick the triple that maximizes Macro F1, then evaluate on the **untouched** MIT-BIH test set.

**Selection rule (sprint plan):** pick best by **cascade Macro F1 on MIT-BIH test**, not val. Val is not trustworthy (v9.4 Phase B confirmed this).


In [ ]:
# ── Joint sweep on val ────────────────────────────────────────────────────────
gate_probs_val_full = gate_v93.predict([X_val, X_rr_val], batch_size=256, verbose=0).flatten()
v_pv, s_pv = sv_v95.predict([X_val, X_rr_val], batch_size=256, verbose=0)
v_pv = v_pv.flatten(); s_pv = s_pv.flatten()
df_sweep = joint_threshold_sweep(gate_probs_val_full, v_pv, s_pv, y_val)
best = df_sweep.loc[df_sweep['macro_f1'].idxmax()]
V95_GATE_THR = float(best['gate_thr'])
V95_V_THR    = float(best['v_thr'])
V95_S_THR    = float(best['s_thr'])
print(f"Best val triple: gate={V95_GATE_THR:.3f}, V={V95_V_THR:.2f}, S={V95_S_THR:.2f}")
print(f"  val macro_f1={float(best['macro_f1']):.4f}, "
      f"S recall={float(best['s_recall']):.3f}, S prec={float(best['s_precision']):.3f}")

# ── Eval on MIT-BIH test (untouched) ─────────────────────────────────────────
v95_test_metrics = eval_cascade(gate_v93, sv_v95, X_test, X_rr_test, y_test,
                                 V95_GATE_THR, V95_V_THR, V95_S_THR,
                                 le, "v9.5 PTB-XL (test)")

print(f"\n{'='*80}")
print(f"v9.5 MIT-BIH TEST RESULTS")
print(f"{'='*80}")
print(f"  Thresholds: gate={V95_GATE_THR:.3f}, V={V95_V_THR:.2f}, S={V95_S_THR:.2f}")
for cls in le.classes_:
    print(f"  {cls}: F1={v95_test_metrics[f'{cls}_f1']:.3f}  "
          f"Se={v95_test_metrics[f'{cls}_recall']:.3f}  "
          f"P+={v95_test_metrics[f'{cls}_precision']:.3f}  "
          f"n={v95_test_metrics[f'{cls}_total']}")
print(f"  Macro F1: {v95_test_metrics['macro_f1']:.4f}")

# ── Compare to v9.3 baseline ─────────────────────────────────────────────────
print(f"\n{'='*80}")
print(f"v9.5 vs v9.3 (Δ from baseline)")
print(f"{'='*80}")
print(f"{'Metric':<14} {'v9.3':>8} {'v9.5':>8} {'Δ':>9}")
print(f"{'-'*40}")
for k, label in [('S_f1','S F1'), ('S_recall','S recall'),
                  ('S_precision','S prec'), ('V_f1','V F1'),
                  ('V_recall','V recall'), ('N_f1','N F1'),
                  ('macro_f1','Macro F1')]:
    a = v93_test_metrics[k]; b = v95_test_metrics[k]
    print(f"  {label:<12} {a:>8.4f} {b:>8.4f} {b-a:>+9.4f}")

# ── Day-2 decision tree (sprint plan) ────────────────────────────────────────
v95_s_recall = v95_test_metrics['S_recall']
v95_val_s_recall = float(best['s_recall'])
gap = v95_val_s_recall - v95_s_recall

print(f"\n{'='*80}")
print(f"DAY-2 DECISION TREE (sprint plan)")
print(f"{'='*80}")
print(f"  Val S recall:  {v95_val_s_recall:.3f}")
print(f"  Test S recall: {v95_s_recall:.3f}")
print(f"  Val→Test gap:  {gap:.3f} (v9.3 gap was ~0.52)")
print()
if v95_s_recall > 0.25 and gap < 0.40:
    print("  → CONFIRMED: Patient generalization hypothesis.")
    print("    130-sample + PTB-XL is the base model for the week.")
    print("    Day 3: expand PTB-XL extraction (target 10,000 beats) + Extended RR features.")
    print("    Expected ceiling with more data: S F1 0.30-0.35.")
elif v95_s_recall < 0.20 and gap > 0.50:
    print("  → FAILED: PTB-XL 500→250 resampling or lead mismatch too severe.")
    print("    Audit 100 PTB-XL PAC beats visually (check QRS distortion).")
    print("    Try extracting from PTB-XL's records100/ (100 Hz) instead.")
    print("    If still failing, S is morphologically unlearnable on single-lead.")
else:
    print("  → MODEST: Data helps but needs volume.")
    print("    Expand PTB-XL extraction to 2000+ records.")
    print("    Proceed to Day 3 feature experiments on top of PTB-XL data.")

ALL_RESULTS.append({'experiment': 'v9.5_ptbxl', **v95_test_metrics,
                    'gate_thr': V95_GATE_THR, 'v_thr': V95_V_THR, 's_thr': V95_S_THR})


## INCART cross-check (external validation)

INCART = St Petersburg INCART 12-lead Arrhythmia Database (75 records, 257 Hz, beat-level annotations). Completely held-out — used as a generalization probe.

**Sprint plan rule:** If INCART Macro F1 is within 0.05 of MIT-BIH test Macro F1, the model generalizes. If INCART is much worse, the model is still overfitting to MIT-BIH morphology.

Cell runs gracefully (skips with a warning) if the INCART path is not present locally.


In [ ]:
def load_incart_records(db_path, source_fs=INCART_FS, target_fs=TARGET_FS,
                        window=WINDOW_LEN, prematurity_threshold=CLEANING_THRESHOLD):
    """Load INCART records, resample 257→250 Hz, extract beats, compute 7 RR features.

    Returns (beats, rr, labels, recs) in the same format as MIT-BIH/SVDB.
    """
    all_beats, all_rr, all_labels, all_recs = [], [], [], []
    if not os.path.isdir(db_path):
        return (np.empty((0, window, 2), dtype=np.float32),
                np.empty((0, 7), dtype=np.float32),
                np.array([], dtype=object), np.array([], dtype=object))

    hea_files = sorted(glob.glob(os.path.join(db_path, '*.hea')))
    rec_ids = [os.path.splitext(os.path.basename(f))[0] for f in hea_files]
    half = window // 2

    for rec_id in rec_ids:
        try:
            record = wfdb.rdrecord(f'{db_path}/{rec_id}')
            annotation = wfdb.rdann(f'{db_path}/{rec_id}', 'atr')
            n_channels = record.p_signal.shape[1]
            # 257 -> 250 Hz polyphase (coprime — use up=250, down=257)
            sig0 = resample_poly(record.p_signal[:, 0], up=target_fs, down=source_fs)
            sig1 = resample_poly(record.p_signal[:, 1], up=target_fs, down=source_fs) \
                   if n_channels > 1 else sig0
            ecg_ch0 = rolling_window_normalize(sig0, target_fs)
            ecg_ch1 = rolling_window_normalize(sig1, target_fs)

            peak_idx = annotation.sample
            if source_fs != target_fs:
                peak_idx = np.round(peak_idx * target_fs / source_fs).astype(int)
            peaks_sec = peak_idx / target_fs

            for k, sym in enumerate(annotation.symbol):
                if sym not in BEAT_MAP: continue
                aami = BEAT_MAP[sym]
                if aami not in CLASSES_TO_USE: continue
                center = peak_idx[k]
                lo = center - half; hi = center + half
                if lo < 0 or hi >= len(ecg_ch0): continue
                beat = np.stack([ecg_ch0[lo:hi], ecg_ch1[lo:hi]], axis=-1).astype(np.float32)
                rr = compute_rr_features(peaks_sec, k)
                all_beats.append(beat); all_rr.append(rr)
                all_labels.append(aami); all_recs.append(f'incart_{rec_id}')
        except Exception as e:
            print(f"  [skip] incart_{rec_id}: {e}")

    if not all_beats:
        return (np.empty((0, window, 2), dtype=np.float32),
                np.empty((0, 7), dtype=np.float32),
                np.array([], dtype=object), np.array([], dtype=object))
    return (np.stack(all_beats), np.stack(all_rr),
            np.array(all_labels, dtype=object), np.array(all_recs, dtype=object))

incart_metrics = None
if os.path.isdir(INCART_PATH):
    print(f"Loading INCART from {INCART_PATH} ...")
    inc_b, inc_rr, inc_l, inc_rec = load_incart_records(INCART_PATH)
    # Apply same 0.95 cleaning as MIT-BIH
    premat_inc = inc_rr[:, 6]
    relabel_inc = (inc_l == 'S') & (premat_inc >= CLEANING_THRESHOLD)
    inc_l_clean = inc_l.copy(); inc_l_clean[relabel_inc] = 'N'
    y_inc = le.transform(inc_l_clean)
    X_rr_inc_norm = (inc_rr - RR_MEAN) / RR_STD
    print(f"  INCART beats: {len(inc_b)}, S (cleaned)={int(np.sum(y_inc==s_idx))}, "
          f"V={int(np.sum(y_inc==v_idx))}, N={int(np.sum(y_inc==n_idx))}")

    incart_metrics = eval_cascade(gate_v93, sv_v95, inc_b, X_rr_inc_norm, y_inc,
                                   V95_GATE_THR, V95_V_THR, V95_S_THR,
                                   le, "v9.5 INCART")
    print(f"\n  v9.5 INCART: macro_f1={incart_metrics['macro_f1']:.4f}, "
          f"S F1={incart_metrics['S_f1']:.3f}, S recall={incart_metrics['S_recall']:.3f}, "
          f"V F1={incart_metrics['V_f1']:.3f}")
    delta = incart_metrics['macro_f1'] - v95_test_metrics['macro_f1']
    print(f"  Δ (INCART - MIT-BIH test) macro_f1 = {delta:+.4f}")
    if abs(delta) <= 0.05:
        print("  ✓ Within 0.05 → model generalizes (per sprint plan rule).")
    else:
        print(f"  ⚠ Outside 0.05 → model is still overfitting to MIT-BIH morphology.")

    # Per-patient S regime analysis (sprint plan: did `both_low` drop from 11/26?)
    print(f"\n  Per-patient S regime (INCART):")
    inc_recs_unique = sorted(set(inc_rec))
    both_low = both_ok = no_target = 0
    for r in inc_recs_unique:
        m = inc_rec == r
        if m.sum() == 0: continue
        y_t = y_inc[m]
        n_s = int(np.sum(y_t == s_idx))
        if n_s == 0:
            no_target += 1; continue
        # Reuse the predictions made inside eval_cascade? Re-predict to get per-patient.
        gate_p = gate_v93.predict([inc_b[m], X_rr_inc_norm[m]], batch_size=256, verbose=0).flatten()
        pass_m = gate_p > V95_GATE_THR
        y_p = np.full(len(y_t), n_idx, dtype=int)
        if pass_m.any():
            v_p, s_p = sv_v95.predict([inc_b[m][pass_m], X_rr_inc_norm[m][pass_m]],
                                       batch_size=256, verbose=0)
            v_p = v_p.flatten(); s_p = s_p.flatten()
            routed = np.full(len(v_p), n_idx, dtype=int)
            v_fire = v_p > V95_V_THR; routed[v_fire] = v_idx
            s_fire = (~v_fire) & (s_p > V95_S_THR); routed[s_fire] = s_idx
            y_p[pass_m] = routed
        s_rec = int(np.sum((y_t == s_idx) & (y_p == s_idx))) / max(n_s, 1)
        if s_rec < 0.30: both_low += 1
        elif s_rec >= 0.50: both_ok += 1
    print(f"    both_low (S rec < 0.30): {both_low}")
    print(f"    both_ok  (S rec ≥ 0.50): {both_ok}")
    print(f"    no_target (no S in record): {no_target}")
else:
    print(f"⚠ INCART path not found: {INCART_PATH}")
    print(f"  Skipping INCART cross-check. metrics.json incart_test will be null.")
    print(f"  Download from https://physionet.org/content/incartdb/1.0.0/ if you want this check.")


## Save `metrics.json` + summary

Persist the **exact schema from the sprint plan** so this run can be compared apples-to-apples with v9.6 / v9.7.


In [ ]:
# ── metrics.json (sprint plan schema) ─────────────────────────────────────────
metrics = {
    "version": "v9.5_ptbxl",
    "date": datetime.now().strftime("%Y-%m-%d"),
    "seed": SEED,
    "window_samples": WINDOW_LEN,
    "cleaning_threshold": CLEANING_THRESHOLD,
    "n_share_target": N_SHARE_TARGET,
    "dataset": {
        "mitbih_train_s": int(np.sum((y_raw_all_clean == 'S') & train_mask
                                      & np.isin(recs_all, [f'mitbih_{r}' for r in MITBIH_TRAIN_RECORDS]))),
        "svdb_train_s":   int(np.sum((y_raw_all_clean == 'S') & train_mask
                                      & np.isin(recs_all, [f'svdb_{r}' for r in SVDB_RECORDS]))),
        "ptbxl_train_s":  int(len(ptbxl_beats)),
    },
    "thresholds": {
        "gate": V95_GATE_THR,
        "v": V95_V_THR,
        "s": V95_S_THR,
    },
    "mitbih_test": {
        "n_f1":       float(v95_test_metrics['N_f1']),
        "s_f1":       float(v95_test_metrics['S_f1']),
        "s_recall":   float(v95_test_metrics['S_recall']),
        "s_precision":float(v95_test_metrics['S_precision']),
        "v_f1":       float(v95_test_metrics['V_f1']),
        "v_recall":   float(v95_test_metrics['V_recall']),
        "macro_f1":   float(v95_test_metrics['macro_f1']),
    },
    "incart_test": (
        {"macro_f1": float(incart_metrics['macro_f1']),
         "s_f1": float(incart_metrics['S_f1']),
         "s_recall": float(incart_metrics['S_recall']),
         "v_f1": float(incart_metrics['V_f1'])}
        if incart_metrics is not None else None
    ),
    "per_patient_s_regime": {
        # Filled from INCART analysis if INCART ran, else null.
        "both_low": int(both_low) if incart_metrics is not None else None,
        "both_ok":  int(both_ok)  if incart_metrics is not None else None,
        "no_target":int(no_target) if incart_metrics is not None else None,
    },
    "svdb_audit": {
        "hea_files_found": int(svdb_count),
        "expected": 78,
        "missing_records": sorted(set(range(800, 895)) -
                                   {int(os.path.basename(f).split('.')[0])
                                    for f in svdb_hea_files}) if svdb_count < 78 else [],
    },
    "ptbxl_extraction": {
        "records_processed": int(len(record_ids)) if 'record_ids' in globals() else 0,
        "records_yielded_beats": int(len(per_record_yield)) if 'per_record_yield' in globals() else 0,
        "total_pac_beats": int(len(ptbxl_beats)),
        "yield_checkpoint_passed": bool(len(ptbxl_beats) >= 1000),
        "detector": "wfdb.processing.xqrs_detect (FALLBACK — replace with your Pan-Tompkins)",
    },
    "v93_baseline_mitbih_test": {
        "n_f1":       float(v93_test_metrics['N_f1']),
        "s_f1":       float(v93_test_metrics['S_f1']),
        "s_recall":   float(v93_test_metrics['S_recall']),
        "s_precision":float(v93_test_metrics['S_precision']),
        "v_f1":       float(v93_test_metrics['V_f1']),
        "v_recall":   float(v93_test_metrics['V_recall']),
        "macro_f1":   float(v93_test_metrics['macro_f1']),
    },
}

with open(f'{OUTPUTS_V95}/metrics.json', 'w') as f:
    json.dump(metrics, f, indent=2)
print(f"✓ Saved: {OUTPUTS_V95}/metrics.json")
print(json.dumps(metrics, indent=2))

# ── v95_summary.json (richer — for KB Section 44) ────────────────────────────
summary = {
    "timestamp": datetime.now().isoformat(),
    "version": "v9.5_ptbxl",
    "sprint_day": 1,
    "config": {
        "window_samples": WINDOW_LEN,
        "cleaning_threshold": CLEANING_THRESHOLD,
        "n_share_target": N_SHARE_TARGET,
        "seed": SEED,
        "epochs": 60,
        "batch_size": 256,
        "early_stop_patience": 12,
        "gate_model": "v9.3 (frozen)",
        "sv_architecture": "Conv2D(16,7)->Conv2D(32,5)->Conv2D(48,5)->Conv2D(48,3) + Dense(16,8) RR branch + Dense(32) merge + V/S heads",
        "sv_params": int(sv_v95.count_params()),
    },
    "ptbxl_extraction": metrics["ptbxl_extraction"],
    "sv_train_composition": {
        "N": int(final_counts.get(n_idx, 0)),
        "S": int(final_counts.get(s_idx, 0)),
        "V": int(final_counts.get(v_idx, 0)),
        "total": int(len(sv_y_aug)),
    },
    "v93_baseline_mitbih_test": metrics["v93_baseline_mitbih_test"],
    "v95_mitbih_test": metrics["mitbih_test"],
    "v95_incart_test": metrics["incart_test"],
    "thresholds": metrics["thresholds"],
    "all_results": ALL_RESULTS,
    "artifacts": {
        "sv_model": f"{OUTPUTS_V95}/sv_v95.keras",
        "metrics_json": f"{OUTPUTS_V95}/metrics.json",
        "ptbxl_beats": f"{OUTPUTS_V95}/ptbxl_pac_beats.npz",
        "sv_train_augmented": f"{OUTPUTS_V95}/sv_train_augmented.npz",
        "spotcheck_png": f"{OUTPUTS_V95}/ptbxl_spotcheck.png",
    },
    "next_steps": [
        "Day 2: read v9.5 metrics, decide branch (CONFIRMED / FAILED / MODEST)",
        "If CONFIRMED: expand PTB-XL extraction to 10,000 beats overnight",
        "Day 3: add 3 extended RR features (P-wave energy, post-beat energy, RR acceleration)",
        "Day 4: INT8 quantization + best model selection",
        "Day 6: KB Section 44 update",
    ],
}
with open(f'{OUTPUTS_V95}/v95_summary.json', 'w') as f:
    json.dump(summary, f, indent=2, default=str)
print(f"\n✓ Saved: {OUTPUTS_V95}/v95_summary.json")

# ── Final comparison table ───────────────────────────────────────────────────
print(f"\n{'='*100}")
print(f"v9.5 FINAL SUMMARY — PTB-XL PAC AUGMENTATION (Day 1)")
print(f"{'='*100}")
df = pd.DataFrame(ALL_RESULTS)
display_cols = ['experiment', 'macro_f1', 'S_f1', 'S_recall', 'S_precision', 'V_f1', 'V_recall']
df_d = df[display_cols].copy()
for c in display_cols[1:]:
    df_d[c] = df_d[c].astype(float).round(4)
print(df_d.to_string(index=False))

print(f"\n{'─'*100}")
print(f"{'Experiment':<25} {'S F1':>8} {'Δ from v9.3':>12} {'S recall':>10} {'S prec':>8} {'Macro F1':>10}")
print('─' * 75)
v93_sf1 = v93_test_metrics['S_f1']
for _, row in df.iterrows():
    d_s = float(row['S_f1']) - v93_sf1
    marker = '★' if d_s > 0.03 else ('✓' if d_s > 0 else ' ')
    print(f"  {marker} {row['experiment']:<21} {float(row['S_f1']):>8.4f} {d_s:>+12.4f} "
          f"{float(row['S_recall']):>10.4f} {float(row['S_precision']):>8.4f} "
          f"{float(row['macro_f1']):>10.4f}")

print(f"\nArtifacts ready in {OUTPUTS_V95}/:")
print(f"  - sv_v95.keras         (best SV head by val_combined_score)")
print(f"  - metrics.json         (sprint plan schema — for KB / firmware handoff)")
print(f"  - v95_summary.json     (richer summary for KB Section 44)")
print(f"  - ptbxl_pac_beats.npz  (extracted PTB-XL PAC beats — reuse for v9.6)")
print(f"  - sv_train_augmented.npz (final augmented SV training set)")
print(f"  - ptbxl_spotcheck.png  (50-beat visual QA)")
print(f"\nNext: Day-2 decision-tree branch is printed above. Follow that branch for Day 3.")
